# Checking output files

In [3]:
from datetime import datetime, timedelta
import calendar
import collections
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
import sys

import glob
import pandas as pd
import xarray as xr
from IPython.display import display, HTML

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


my_dir = "/g/data/eg3/spr548/projects/"
sys.path.append(os.path.join(my_dir, "nesp_bff")+os.sep)
from nathers import location_details

# import utils
# from utils import locations, model_dict, vars_1hr, vars_day

In [41]:
step = "step3"
show_only_if_flagged = True

#==========================================
root_dir = "/g/data/eg3/nesp_bff/"
# location = "Sydney"
# scenario = "ssp370"
# model = "CESM2"
# time_period = "2041-2060"
vars_to_summarise_step2 = ["tas","huss","sfcWind","psl","rsds","rsdsdir","rsdsdif"]
vars_to_summarise_step3 = ["tas","twbt","huss","psl","wind_speed","wind_dir","clt","rsds","rsdsdir","rsdsdif"]
locations = ["Darwin","Cairns","Brisbane","Longreach","Mildura","Adelaide","Perth","Sydney","Melbourne","Canberra","Hobart"]
vars_maybe_drop = ["time_offset", "round_method", "crs", "lat", "lon"]

input_dir = f"{root_dir}step3_calc_missing_vars/" if step == "step3" else f"{root_dir}step2_qdc_scaling/BARPA-R/"
vars_to_summarise = vars_to_summarise_step3 if step == "step3" else vars_to_summarise_step2

In [42]:
files = glob.glob(f"{input_dir}*.nc")
len(files)

462

In [43]:
# # Old checkds
# qc_rows = []
# errors = []

# for file in sorted(files):
#     base = file.split("/")[-1]
#     parts = base.split("_")

#     loc = parts[0] if len(parts) > 0 else None
#     model_ = parts[2] if len(parts) > 2 else None
#     ssp = parts[3] if len(parts) > 3 else None
#     time_period_ = parts[9] if len(parts) > 9 else None

#     header = f"{loc}: {model_}, {ssp}, {time_period_}"
#     print(f"==================== {header} ====================")

#     try:
#         with xr.open_dataset(file) as da:
#             df = (
#                 da.drop_vars([v for v in vars_maybe_drop if v in da.variables])
#                   [vars_to_summarise]
#                   .to_dataframe()
#             )

#         desc = df.describe()

#         flagged = df.isna().any().any() or (df.nunique(dropna=True) <= 1).any()

#         if (not show_only_if_flagged) or flagged:
#             print("⚠️ Flagged" if flagged else "OK")
#             display(HTML('<div style="overflow-x:auto; max-width:100%;">'))
#             display(desc)
#             display(HTML("</div>"))

#         qc_rows.append({
#             "file": base,
#             "loc": loc,
#             "model": model_,
#             "ssp": ssp,
#             "time_period": time_period_,
#             "n_min": int(desc.loc["count"].min()),
#             "any_nan": bool(df.isna().any().any()),
#             "any_const": bool((df.nunique(dropna=True) <= 1).any()),
#             "rsds_min": float(desc.loc["min", "rsds"]) if "rsds" in desc.columns else None,
#             "rsds_max": float(desc.loc["max", "rsds"]) if "rsds" in desc.columns else None,
#         })

#     except Exception as e:
#         errors.append({
#             "file": base,
#             "loc": loc,
#             "model": model_,
#             "ssp": ssp,
#             "time_period": time_period_,
#             "error": repr(e),
#         })

# qc_df = pd.DataFrame(qc_rows)
# err_df = pd.DataFrame(errors)

# print("\n=== QC summary (all files) ===")
# display(qc_df)

# if not err_df.empty:
#     print("\n=== Errors ===")
#     display(err_df)

In [44]:
vars_maybe_drop = ["time_offset", "round_method", "crs", "lat", "lon"]
RANGES = {
   # temps: keep the K→C heuristic because files might still be in K
   "tas":  {"min": -60, "max":  60, "units": "C_or_K"},
   "twbt": {"min": -60, "max":  60, "units": "C_or_K"},
   # specific humidity (g/kg)
   "huss": {"min": 0.0, "max": 40., "units": "g/kg"},
   # sea level pressure in kPa (typical ~ 90–110 kPa; give generous bounds)
   "psl":  {"min": 80.0, "max": 110.0, "units": "kPa"},
   # wind speed (m/s)
   "wind_speed": {"min": 0.0, "max": 60.0, "units": "m/s"},
   # 16-point direction code (0..16). If you treat 0 as calm, keep it allowed.
   "wind_dir": {"min": 0.0, "max": 16.0, "units": "dir16"},
   # cloud cover in oktas (0..8)
   "clt": {"min": 0.0, "max": 8.0, "units": "oktas"},
   # radiation (W/m²)
   "rsds":    {"min": 0.0, "max": 1300.0, "units": "W/m2"},
   "rsdsdir": {"min": 0.0, "max": 1300.0, "units": "W/m2"},
   "rsdsdif": {"min": 0.0, "max": 1300.0, "units": "W/m2"},
}

def _convert_for_check(var, vmin, vmax):
    rule = RANGES.get(var)
    if rule is None:
        return vmin, vmax, None, None, None
    rmin, rmax = rule["min"], rule["max"]
    note = None
    # Temperature heuristic: if values look like Kelvin, convert to Celsius for checking
    if var in ("tas", "twbt"):
        if np.isfinite(vmax) and vmax > 150:  # likely Kelvin
            vmin = vmin - 273.15
            vmax = vmax - 273.15
            note = "converted K→C (heuristic)"
    return vmin, vmax, rmin, rmax, note

def flag_df(df, desc, vars_to_check):
   reasons = []
   # NaN + constant checks (overall)
   if df[vars_to_check].isna().any().any():
       reasons.append("has NaNs")
   if (df[vars_to_check].nunique(dropna=True) <= 1).any():
       const_vars = list(df[vars_to_check].columns[(df[vars_to_check].nunique(dropna=True) <= 1)])
       reasons.append(f"constant: {const_vars[:5]}" + ("..." if len(const_vars) > 5 else ""))
   # Range checks per variable
   for v in vars_to_check:
       if v not in desc.columns:
           reasons.append(f"missing column: {v}")
           continue
       vmin = desc.loc["min", v]
       vmax = desc.loc["max", v]
       vmin2, vmax2, rmin, rmax, note = _convert_for_check(v, vmin, vmax)
       if rmin is None:
           continue
       out_low = np.isfinite(vmin2) and (vmin2 < rmin)
       out_high = np.isfinite(vmax2) and (vmax2 > rmax)
       if out_low or out_high:
           msg = f"{v} out of range [{rmin}, {rmax}]"
           if out_low:
               msg += f" (min={vmin2:.3g})"
           if out_high:
               msg += f" (max={vmax2:.3g})"
           if note:
               msg += f" [{note}]"
           reasons.append(msg)
   return reasons

qc_rows, errors = [], []
for file in sorted(files):
   base = file.split("/")[-1]
   parts = base.split("_")
   loc = parts[0] if len(parts) > 0 else None
   model_ = parts[2] if len(parts) > 2 else None
   ssp = parts[3] if len(parts) > 3 else None
   time_period_ = parts[9] if len(parts) > 9 else None
   header = f"{loc}: {model_}, {ssp}, {time_period_}"
   print(f"==================== {header} ====================")
   try:
       with xr.open_dataset(file) as da:
           df = (
               da.drop_vars([v for v in vars_maybe_drop if v in da.variables])
                 [vars_to_summarise]
                 .to_dataframe()
           )
       desc = df.describe()
       reasons = flag_df(df, desc, vars_to_summarise)
       flagged = len(reasons) > 0
       if (not show_only_if_flagged) or flagged:
           print("⚠️ Flagged" if flagged else "OK")
           if flagged:
               print("Reasons:", "; ".join(reasons[:6]) + (" ..." if len(reasons) > 6 else ""))
           display(HTML('<div style="overflow-x:auto; max-width:100%;">'))
           display(desc)
           display(HTML("</div>"))
       qc_rows.append({
           "file": base,
           "loc": loc,
           "model": model_,
           "ssp": ssp,
           "time_period": time_period_,
           "flagged": flagged,
           "flag_reason": "; ".join(reasons[:8]) + (" ..." if len(reasons) > 8 else ""),
           "n_min": int(desc.loc["count"].min()),
           "any_nan": bool(df.isna().any().any()),
           "any_const": bool((df.nunique(dropna=True) <= 1).any()),
       })
   except Exception as e:
       errors.append({
           "file": base,
           "loc": loc,
           "model": model_,
           "ssp": ssp,
           "time_period": time_period_,
           "error": repr(e),
       })
qc_df = pd.DataFrame(qc_rows)
err_df = pd.DataFrame(errors)
display(qc_df.sort_values(["flagged", "loc", "model"], ascending=[False, True, True]))
if not err_df.empty:
   display(err_df)

==================== Adelaide: ACCESS-CM2, ssp126, 2021-2040 ====================
==================== Adelaide: ACCESS-CM2, ssp126, 2041-2060 ====================
==================== Adelaide: ACCESS-CM2, ssp126, 2061-2080 ====================
==================== Adelaide: ACCESS-CM2, ssp370, 2021-2040 ====================
==================== Adelaide: ACCESS-CM2, ssp370, 2041-2060 ====================
==================== Adelaide: ACCESS-CM2, ssp370, 2061-2080 ====================
==================== Adelaide: ACCESS-ESM1-5, ssp126, 2021-2040 ====================
==================== Adelaide: ACCESS-ESM1-5, ssp126, 2041-2060 ====================
==================== Adelaide: ACCESS-ESM1-5, ssp126, 2061-2080 ====================
==================== Adelaide: ACCESS-ESM1-5, ssp370, 2021-2040 ====================
==================== Adelaide: ACCESS-ESM1-5, ssp370, 2041-2060 ====================
==================== Adelaide: ACCESS-ESM1-5, ssp370, 2061-2080 ===================

,tas,twbt,huss,psl,wind_speed,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000
mean,25.411657,22.155165,15.471016,101.204277,4.286536,6.708042,4.805354,215.706558,224.774323,61.249191
std,3.412515,3.030259,3.333262,0.429621,2.242982,3.400200,2.418833,300.816040,282.856384,79.482162
min,9.387857,6.731903,3.296445,98.392296,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,23.154748,20.118215,13.040805,100.916870,2.923209,5.000000,2.000000,0.000000,0.000000,0.000000
50%,25.565057,22.458327,15.510452,101.232506,4.318677,7.000000,5.000000,0.000000,0.000000,0.000000
75%,27.746741,24.538730,18.111002,101.525482,5.660056,8.000000,7.000000,397.382576,465.965485,118.519346
max,40.765678,30.511023,27.007463,102.409813,22.466087,16.000000,8.000000,1470.000000,1092.411133,444.537598


==================== Cairns: ACCESS-CM2, ssp126, 2041-2060 ====================
⚠️ Flagged
Reasons: rsds out of range [0.0, 1300.0] (max=1.47e+03)


,tas,twbt,huss,psl,wind_speed,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000
mean,25.809458,22.312113,15.553292,101.225014,4.432554,6.708042,4.805354,220.473816,233.914520,60.275497
std,3.597321,3.179941,3.476262,0.441653,2.277860,3.400200,2.418833,303.830566,290.730042,77.356110
min,9.227594,5.889141,2.749305,98.157471,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,23.508805,20.234712,13.096956,100.933319,3.069167,5.000000,2.000000,0.000000,0.000000,0.000000
50%,25.981287,22.662785,15.607604,101.260548,4.499134,7.000000,5.000000,0.000000,0.000000,0.000000
75%,28.239904,24.812951,18.326318,101.546516,5.862076,8.000000,7.000000,413.100533,487.058792,118.022793
max,40.455910,31.310020,27.179726,102.490105,20.816994,16.000000,8.000000,1470.000000,1104.624878,420.747681


==================== Cairns: ACCESS-CM2, ssp126, 2061-2080 ====================
⚠️ Flagged
Reasons: rsds out of range [0.0, 1300.0] (max=1.47e+03)


...

==================== Sydney: NorESM2-MM, ssp370, 2041-2060 ====================
⚠️ Flagged
Reasons: wind_speed out of range [0.0, 60.0] (max=83.3)


,tas,twbt,huss,psl,wind_speed,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000
mean,19.393860,15.389584,9.529109,101.717720,5.312337,8.790234,4.708213,189.813889,219.535156,59.015137
std,5.183980,4.596050,3.512263,0.710977,2.877304,4.883113,2.710749,275.624451,277.890167,83.941505
min,3.344633,1.965032,1.002830,98.729424,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,15.854742,11.889095,6.665365,101.241888,3.118707,5.000000,2.000000,0.000000,0.000000,0.000000
50%,19.636237,15.733582,9.345748,101.731232,4.778315,9.000000,5.000000,0.000000,0.000000,0.000000
75%,22.750093,19.096116,12.203608,102.213982,7.215644,13.000000,8.000000,333.052643,458.231560,103.102039
max,48.562614,27.148521,20.826513,104.017967,83.335602,16.000000,8.000000,1121.113525,819.049805,359.027405


==================== Sydney: NorESM2-MM, ssp370, 2061-2080 ====================
⚠️ Flagged
Reasons: wind_speed out of range [0.0, 60.0] (max=82.1)


,tas,twbt,huss,psl,wind_speed,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000
mean,19.999115,16.088320,10.061976,101.763756,5.286878,8.790234,4.708213,188.246948,215.873856,59.495155
std,5.136320,4.553900,3.581531,0.726449,2.857944,4.883113,2.710749,273.610565,273.596619,84.549782
min,3.560622,2.241216,1.049449,98.270203,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,16.490033,12.694622,7.203699,101.289902,3.104640,5.000000,2.000000,0.000000,0.000000,0.000000
50%,20.248775,16.510550,9.990279,101.783882,4.754998,9.000000,5.000000,0.000000,0.000000,0.000000
75%,23.385255,19.741238,12.789201,102.277208,7.218602,13.000000,8.000000,329.770927,448.833176,103.640825
max,47.508217,27.678799,22.023497,104.237320,82.115776,16.000000,8.000000,1115.096069,814.281982,368.808624


,file,loc,model,ssp,time_period,flagged,flag_reason,n_min,any_nan,any_const
84,Cairns_AUS-15_ACCESS-CM2_ssp126_r4i1p1f1_BOM_B...,Cairns,ACCESS-CM2,ssp126,2021-2040,True,"rsds out of range [0.0, 1300.0] (max=1.47e+03)",175200,False,False
85,Cairns_AUS-15_ACCESS-CM2_ssp126_r4i1p1f1_BOM_B...,Cairns,ACCESS-CM2,ssp126,2041-2060,True,"rsds out of range [0.0, 1300.0] (max=1.47e+03)",175200,False,False
86,Cairns_AUS-15_ACCESS-CM2_ssp126_r4i1p1f1_BOM_B...,Cairns,ACCESS-CM2,ssp126,2061-2080,True,"rsds out of range [0.0, 1300.0] (max=1.47e+03)",175200,False,False
87,Cairns_AUS-15_ACCESS-CM2_ssp370_r4i1p1f1_BOM_B...,Cairns,ACCESS-CM2,ssp370,2021-2040,True,"rsds out of range [0.0, 1300.0] (max=1.47e+03)",175200,False,False
88,Cairns_AUS-15_ACCESS-CM2_ssp370_r4i1p1f1_BOM_B...,Cairns,ACCESS-CM2,ssp370,2041-2060,True,"rsds out of range [0.0, 1300.0] (max=1.47e+03)",175200,False,False
...,...,...,...,...,...,...,...,...,...,...
415,Perth_AUS-15_NorESM2-MM_ssp126_r1i1p1f1_BOM_BA...,Perth,NorESM2-MM,ssp126,2041-2060,False,,175200,False,False
416,Perth_AUS-15_NorESM2-MM_ssp126_r1i1p1f1_BOM_BA...,Perth,NorESM2-MM,ssp126,2061-2080,False,,175200,False,False
417,Perth_AUS-15_NorESM2-MM_ssp370_r1i1p1f1_BOM_BA...,Perth,NorESM2-MM,ssp370,2021-2040,False,,175200,False,False
418,Perth_AUS-15_NorESM2-MM_ssp370_r1i1p1f1_BOM_BA...,Perth,NorESM2-MM,ssp370,2041-2060,False,,175200,False,False


In [39]:
qc_rows

[]

# Check NatHERS

In [18]:
show_only_if_flagged = False

#==========================================
nathers_dir = "/g/data/eg3/spr548/projects/nesp_bff/data/raw_data/NatHERS/historical/"
# location = "Sydney"
vars_to_summarise_nathers = ["tas","huss","psl","sfcWind","wind_dir","clt","rsds","rsdsdir","rsdsdif"]
locations = ["Darwin","Cairns","Brisbane","Longreach","Mildura","Adelaide","Perth","Sydney","Melbourne","Canberra","Hobart"]
vars_maybe_drop = ["time_offset", "round_method", "crs", "lat", "lon"]

In [19]:
files = glob.glob(f"{nathers_dir}*.nc")
len(files)

11

In [22]:
vars_maybe_drop = ["time_offset", "round_method", "crs", "lat", "lon"]
RANGES_NATHERS = {
   # temps: keep the K→C heuristic because files might still be in K
   "tas":  {"min": -60, "max":  60, "units": "C_or_K"},
   "twbt": {"min": -60, "max":  60, "units": "C_or_K"},
   # specific humidity (kg/kg)
   "huss": {"min": 0.0, "max": 0.04, "units": "kg/kg"},
   # sea level pressure in kPa (typical ~ 90–110 kPa; give generous bounds)
   "psl":  {"min": 80000, "max": 110000, "units": "Pa"},
   # wind speed (m/s)
   "sfcWind": {"min": 0.0, "max": 60.0, "units": "m/s"},
   # 16-point direction code (0..16). If you treat 0 as calm, keep it allowed.
   "wind_dir": {"min": 0.0, "max": 16.0, "units": "dir16"},
   # cloud cover in fraction classes (0..100)
   "clt": {"min": 0.0, "max": 100.0, "units": "%"},
   # radiation (W/m²)
   "rsds":    {"min": 0.0, "max": 1300.0, "units": "W/m2"},
   "rsdsdir": {"min": 0.0, "max": 1300.0, "units": "W/m2"},
   "rsdsdif": {"min": 0.0, "max": 1300.0, "units": "W/m2"},
}

def _convert_for_check(var, vmin, vmax):
    rule = RANGES_NATHERS.get(var)
    if rule is None:
        return vmin, vmax, None, None, None
    rmin, rmax = rule["min"], rule["max"]
    note = None
    # Temperature heuristic: if values look like Kelvin, convert to Celsius for checking
    if var in ("tas", "twbt"):
        if np.isfinite(vmax) and vmax > 150:  # likely Kelvin
            vmin = vmin - 273.15
            vmax = vmax - 273.15
            note = "converted K→C (heuristic)"
    return vmin, vmax, rmin, rmax, note

def flag_df(df, desc, vars_to_check):
   reasons = []
   # NaN + constant checks (overall)
   if df[vars_to_check].isna().any().any():
       reasons.append("has NaNs")
   if (df[vars_to_check].nunique(dropna=True) <= 1).any():
       const_vars = list(df[vars_to_check].columns[(df[vars_to_check].nunique(dropna=True) <= 1)])
       reasons.append(f"constant: {const_vars[:5]}" + ("..." if len(const_vars) > 5 else ""))
   # Range checks per variable
   for v in vars_to_check:
       if v not in desc.columns:
           reasons.append(f"missing column: {v}")
           continue
       vmin = desc.loc["min", v]
       vmax = desc.loc["max", v]
       vmin2, vmax2, rmin, rmax, note = _convert_for_check(v, vmin, vmax)
       if rmin is None:
           continue
       out_low = np.isfinite(vmin2) and (vmin2 < rmin)
       out_high = np.isfinite(vmax2) and (vmax2 > rmax)
       if out_low or out_high:
           msg = f"{v} out of range [{rmin}, {rmax}]"
           if out_low:
               msg += f" (min={vmin2:.3g})"
           if out_high:
               msg += f" (max={vmax2:.3g})"
           if note:
               msg += f" [{note}]"
           reasons.append(msg)
   return reasons

qc_rows, errors = [], []
for file in sorted(files):
   base = file.split("/")[-1]
   parts = base.split("_")
   loc = parts[0] if len(parts) > 0 else None
   model_ = parts[2] if len(parts) > 2 else None
   ssp = parts[3] if len(parts) > 3 else None
   time_period_ = parts[9] if len(parts) > 9 else None
   header = f"{loc}: {model_}, {ssp}, {time_period_}"
   print(f"==================== {header} ====================")
   try:
       with xr.open_dataset(file) as da:
           df = (
               da.drop_vars([v for v in vars_maybe_drop if v in da.variables])
                 [vars_to_summarise_nathers]
                 .to_dataframe()
           )
       desc = df.describe()
       reasons = flag_df(df, desc, vars_to_summarise_nathers)
       flagged = len(reasons) > 0
       if (not show_only_if_flagged) or flagged:
           print("⚠️ Flagged" if flagged else "OK")
           if flagged:
               print("Reasons:", "; ".join(reasons[:6]) + (" ..." if len(reasons) > 6 else ""))
           display(HTML('<div style="overflow-x:auto; max-width:100%;">'))
           display(desc)
           display(HTML("</div>"))
       qc_rows.append({
           "file": base,
           "loc": loc,
           "model": model_,
           "ssp": ssp,
           "time_period": time_period_,
           "flagged": flagged,
           "flag_reason": "; ".join(reasons[:8]) + (" ..." if len(reasons) > 8 else ""),
           "n_min": int(desc.loc["count"].min()),
           "any_nan": bool(df.isna().any().any()),
           "any_const": bool((df.nunique(dropna=True) <= 1).any()),
       })
   except Exception as e:
       errors.append({
           "file": base,
           "loc": loc,
           "model": model_,
           "ssp": ssp,
           "time_period": time_period_,
           "error": repr(e),
       })
qc_df = pd.DataFrame(qc_rows)
err_df = pd.DataFrame(errors)
# display(qc_df.sort_values(["flagged", "loc", "model"], ascending=[False, True, True]))
if not err_df.empty:
   display(err_df)

==================== Adelaide: AD, NatHERS, None ====================
⚠️ Flagged
Reasons: clt out of range [0.0, 100.0] (max=112)


,tas,huss,psl,sfcWind,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000
mean,290.235447,0.006868,101141.887374,3.095014,7.551807,47.187138,195.122306,200.690435,67.456534
std,6.360441,0.001892,690.444434,1.949950,5.005412,33.541743,287.828158,332.992905,104.313754
min,273.760000,0.000400,97700.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,285.760000,0.005600,100700.000000,1.600000,3.000000,12.500000,0.000000,0.000000,0.000000
50%,289.160000,0.006600,101100.000000,3.100000,8.000000,50.000000,0.000000,0.000000,0.000000
75%,293.860000,0.007900,101600.000000,4.300000,11.000000,75.000000,331.000000,274.000000,97.000000
max,318.260000,0.020200,103300.000000,39.700000,16.000000,112.500000,1133.000000,996.000000,733.000000


==================== Brisbane: BR, NatHERS, None ====================
OK


,tas,huss,psl,sfcWind,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,227904.000000,227904.000000,227904.000000,227904.00000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000
mean,293.397539,0.010743,101530.786208,3.21166,5.965306,49.725652,214.592447,227.532435,69.152046
std,5.062948,0.003374,541.671168,2.48058,5.113696,30.958146,302.448910,340.985109,107.799817
min,274.760000,0.001000,99100.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,290.060000,0.008400,101200.000000,0.00000,0.000000,25.000000,0.000000,0.000000,0.000000
50%,293.960000,0.010700,101500.000000,3.10000,6.000000,50.000000,0.000000,0.000000,0.000000
75%,297.160000,0.013100,101900.000000,4.70000,10.000000,75.000000,395.000000,458.250000,99.000000
max,312.160000,0.023500,103300.000000,18.10000,16.000000,100.000000,1138.000000,998.000000,708.000000


==================== Cairns: CN, NatHERS, None ====================
⚠️ Flagged
Reasons: sfcWind out of range [0.0, 60.0] (max=97.8); clt out of range [0.0, 100.0] (max=112); rsds out of range [0.0, 1300.0] (max=1.47e+03)


...

,tas,huss,psl,sfcWind,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000
mean,290.392557,0.006154,101096.453770,3.376092,7.814523,51.602758,211.120551,243.133082,61.173283
std,7.876479,0.002137,680.405195,2.048971,4.777829,32.435900,302.029779,355.681053,95.304209
min,269.760000,0.000100,97900.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,284.560000,0.004700,100600.000000,2.200000,4.000000,25.000000,0.000000,0.000000,0.000000
50%,289.560000,0.005800,101100.000000,3.100000,8.000000,50.000000,1.000000,0.000000,1.000000
75%,295.460000,0.007100,101600.000000,4.700000,11.000000,87.500000,381.000000,512.000000,87.000000
max,319.260000,0.020700,103300.000000,67.800000,16.000000,100.000000,1212.000000,1092.000000,669.000000


==================== Perth: PE, NatHERS, None ====================
OK


,tas,huss,psl,sfcWind,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000
mean,291.444065,0.007744,101410.508811,4.048331,6.586098,39.159306,212.210264,234.947737,62.874991
std,6.581836,0.002227,604.293313,2.688214,4.748247,33.615397,305.001587,352.114248,97.628902
min,272.160000,0.001300,99000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,286.860000,0.006100,101000.000000,2.200000,3.000000,12.500000,0.000000,0.000000,0.000000
50%,290.860000,0.007500,101400.000000,3.600000,6.000000,25.000000,0.000000,0.000000,0.000000
75%,295.360000,0.009100,101800.000000,5.800000,10.000000,75.000000,380.000000,460.000000,92.000000
max,319.160000,0.022400,103600.000000,18.100000,16.000000,100.000000,1149.000000,997.000000,684.000000


==================== Sydney: MA, NatHERS, None ====================
⚠️ Flagged
Reasons: sfcWind out of range [0.0, 60.0] (max=87.5); clt out of range [0.0, 100.0] (max=112)


,tas,huss,psl,sfcWind,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000,227904.000000
mean,291.389073,0.008956,101635.820345,4.425809,7.264857,48.550706,186.517051,191.603719,69.822377
std,4.994831,0.003165,707.614930,3.331891,5.548381,33.913310,274.045984,317.734324,107.536489
min,276.160000,0.001000,98500.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,288.060000,0.006500,101200.000000,2.200000,1.000000,12.500000,0.000000,0.000000,0.000000
50%,291.460000,0.008700,101700.000000,4.200000,8.000000,50.000000,0.000000,0.000000,0.000000
75%,294.760000,0.011300,102100.000000,6.700000,12.000000,87.500000,323.000000,273.000000,100.000000
max,318.660000,0.020000,103900.000000,87.500000,16.000000,112.500000,1125.000000,997.000000,700.000000
